# Training Series: Introduction to eXtended Discontinuous Galerkin methods for multi-phase flow problems <br> - Implementation of an XDG solver for incompressible two-phase flows

In this tutorial/exercise you will implement an XDG-solver for incompressible two-phase flows.

## Problem statement: transient incompressible two-phase flow

The flow within the computational domain $\Omega = \mathfrak{A} \ {\dot\cup} \ \mathfrak{I} \ {\dot\cup} \ \mathfrak{B}, \Omega \in R^2$ is described by the instationary incompressible Navier-Stokes equations 
>$$ \rho_f\left(\frac{\partial \vec{u}}{\partial t}+ \nabla \cdot \left( \vec{u} \otimes \vec{u} \right) \right) = \nabla \cdot \left( -p\mathbf{I} + \mu_f \left( \nabla \vec{u} + \nabla \vec{u}^{\textrm{T}} \right) \right) = \rho_f \vec{f} \quad \textrm{in}\ \Omega \setminus \mathfrak{I} $$

and the continuity equation
>$$ \nabla \cdot \vec{u} = 0 \quad \textrm{in}\ \Omega \setminus \mathfrak{I} $$

The jump conditions are given by
>$$ \llbracket -p \mathbf{I} + \mu_f (\nabla \vec{u} + \nabla \vec{u}^{\textrm{T}}) \rrbracket \vec{n}_{\mathfrak{I}} = \sigma \kappa \vec{n}_{\mathfrak{I}}  $$
>$$ \llbracket \vec{u} \rrbracket  = \vec{0}  $$

At the boundary $\partial \Omega = \partial \Omega_\textrm{D} \cup \partial \Omega_\textrm{N}$ the corresponding boundary conditions read 
>$$ \vec{u} = \vec{u}_\textrm{D} \quad \textrm{on}\ \partial \Omega_\textrm{D} $$
>$$ -p \vec{n}_{\partial \Omega} + \mu_f \left( \nabla \vec{u} + \nabla \vec{u}^{\textrm{T}} \right) \vec{n}_{\partial \Omega} = - p_\textrm{ext} \vec{n}_{\partial \Omega} \quad \textrm{on}\ \partial \Omega_\textrm{N} $$

In the equations above 
 - $\vec{u}(\vec{x}, t)$ is the velocity vector 
 - $p(\vec{x}, t)$ the pressure
 - and $\vec{f}(\vec{x})$ some volume force, e.g. gravity. 
 - The fluid density is denoted by $\rho_f$ and
 - $\mu_f=\rho_f \cdot \nu_f$ is the dynamic viscosity of the fluid.
 - $\sigma$ denotes the surface tension force and
 - $\kappa$ the mean curvature of the interface $\mathfrak{I}$
 



## Implementation of an XDG solver for two-phase flows

In [ ]:
#r "../BoSSSpad/BoSSSpad.dll"
//#r "C:\Program Files (x86)\FDY\BoSSS\bin\Release\net5.0\BoSSSpad.dll"
using System;
using System.Collections.Generic;
using System.Linq;
using ilPSP;
using ilPSP.Utils;
using BoSSS.Platform;
using BoSSS.Foundation;
using BoSSS.Foundation.Grid;
using BoSSS.Foundation.Grid.Classic;
using BoSSS.Foundation.IO;
using BoSSS.Solution;
using BoSSS.Solution.Control;
using BoSSS.Solution.GridImport;
using BoSSS.Solution.Statistic;
using BoSSS.Solution.Utils;
using BoSSS.Solution.Gnuplot;
using BoSSS.Application.BoSSSpad;
using BoSSS.Application.XNSE_Solver;
using static BoSSS.Application.BoSSSpad.BoSSSshell;
Init();

In [ ]:
using System.IO;
using System.Runtime.Serialization;
using BoSSS.Foundation.XDG;
using BoSSS.Foundation.XDG.OperatorFactory;
using BoSSS.Solution.Tecplot;
using BoSSS.Solution.NSECommon;
using BoSSS.Solution.XNSECommon;
using BoSSS.Solution.XNSECommon.Operator;
using BoSSS.Solution.LevelSetTools.SolverWithLevelSetUpdater;

In order to define a `SpatialOperator` for the instationary Navier-Stokes equations, we need to implement the corresponding fluxes at the interface $\mathfrak{I}$.

As an example we are looking at the viscosity terms at the interface, which are discretized in the bulk phases via the SIP method.

In [ ]:
public class ViscosityAtLevelSet : BoSSS.Foundation.XDG.ILevelSetForm, ILevelSetEquationComponentCoefficient {

    public ViscosityAtLevelSet(int SpaceDim, double _muA, double _muB, double _penalty, int _component) {
        this.muA = _muA;
        this.muB = _muB;
        this.m_penalty_base = _penalty;
        this.component = _component;
        this.m_D = SpaceDim;
    }

    double muA;
    double muB;

    int component;
    int m_D;

    /// The flux across the interface is also implemented as an inner flux  
    public double InnerEdgeForm(ref CommonParams inp,
        double[] uA, double[] uB, double[,] Grad_uA, double[,] Grad_uB,
        double vA, double vB, double[] Grad_vA, double[] Grad_vB) {

        double[] N = inp.Normal;
        int D = N.Length;

        double[] Grad_uA_xN = new double[D], Grad_uB_xN = new double[D];
        double Grad_vA_xN = 0, Grad_vB_xN = 0;
        for (int d = 0; d < D; d++) {
            for(int dd = 0; dd < D; dd++) {
                Grad_uA_xN[dd] += Grad_uA[dd, d] * N[d];
                Grad_uB_xN[dd] += Grad_uB[dd, d] * N[d];
            }
            Grad_vA_xN += Grad_vA[d] * N[d];
            Grad_vB_xN += Grad_vB[d] * N[d];
        }

        double Ret = 0.0;

        // SIP implementation
        // ==================

        Ret -= 0.5 * (muA * Grad_uA_xN[component] + muB * Grad_uB_xN[component]) * (vA - vB);       // consistency term
        Ret -= 0.5 * (muA * Grad_vA_xN + muB * Grad_vB_xN) * (uA[component] - uB[component]);       // symmetry term

        // Transposed Term
        for (int i = 0; i < D; i++) {
            Ret -= 0.5 * (muA * Grad_uA[i, component] + muB * Grad_uB[i, component]) * (vA - vB) * N[i];    // consistency term
            Ret -= 0.5 * (muA * Grad_vA[i] + muB * Grad_vB[i]) * (uA[i] - uB[i]) * N[component];            // symmetry term
        }

        double pnlty = this.Penalty(inp.jCellIn, inp.jCellOut);
        double wPenalty = (Math.Abs(muA) > Math.Abs(muB)) ? muA : muB;
        Ret += (uA[component] - uB[component]) * (vA - vB) * pnlty * wPenalty;                          // penalty term

        return Ret;
    }


    /// base multiplier for the penalty computation
    protected double m_penalty_base;

    /// penalty adapted for spatial dimension and DG-degree
    double m_penalty;

    /// computation of penalty parameter according to: $` \mathrm{SafetyFactor} \cdot k^2 / h `$
    protected double Penalty(int jCellIn, int jCellOut) {

        double penaltySizeFactor_A = 1.0 / NegLengthScaleS[jCellIn];
        double penaltySizeFactor_B = 1.0 / PosLengthScaleS[jCellOut];

        double penaltySizeFactor = Math.Max(penaltySizeFactor_A, penaltySizeFactor_B);

        double scaledPenalty = penaltySizeFactor * m_penalty * m_penalty_base;
        if(scaledPenalty.IsNaNorInf())
            throw new ArithmeticException("NaN/Inf detected for penalty parameter.");
        return scaledPenalty;

    }


    MultidimensionalArray PosLengthScaleS;
    MultidimensionalArray NegLengthScaleS;

    
    /// Update of penalty length scales.
    public void CoefficientUpdate(CoefficientSet csA, CoefficientSet csB, int[] DomainDGdeg, int TestDGdeg) {       

        double _D = m_D;
        double _p = DomainDGdeg.Max();

        double penalty_deg_tri = (_p + 1) * (_p + _D) / _D; // formula for triangles/tetras
        double penalty_deg_sqr = (_p + 1.0) * (_p + 1.0); // formula for squares/cubes

        m_penalty = Math.Max(penalty_deg_tri, penalty_deg_sqr); // the conservative choice

        NegLengthScaleS = csA.CellLengthScales;
        PosLengthScaleS = csB.CellLengthScales;
    }

    public int LevelSetIndex {
        get { return 0; }
    }

    public IList<string> ArgumentOrdering {
        get { return VariableNames.VelocityVector(this.m_D); }
    }

    public string PositiveSpecies {
        get { return "B"; }
    }

    public string NegativeSpecies {
        get { return "A"; }
    }

    public TermActivationFlags LevelSetTerms {
        get {
            return TermActivationFlags.UxV | TermActivationFlags.UxGradV | TermActivationFlags.GradUxV;
        }
    }

    public IList<string> ParameterOrdering {
        get { return null;  }
    }

 
}

In addition to extend the bulk discretization across the interface, we need to implement the surace tension force (Laplace-Beltrami formulation), which will be acting as a SpatialOperator on the interface. Thus, we derive from `IVolumeForm` and `IEdgeForm`.

In [ ]:
public class SurfaceTensionForce : IVolumeForm, IEdgeForm {

    int m_comp;
    int m_D;

    /// surface tension coefficient
    double m_sigma;


    public SurfaceTensionForce(int d, int D, double sigma) {
        m_comp = d;
        m_D = D;
        m_sigma = sigma;
    }


    /// VolumeForm on $\mathfrak{I}$
    public double VolumeForm(ref CommonParamsVol cpv, double[] U, double[,] GradU, double V, double[] GradV) {

        double acc = 0;

        double[] Nsurf = SurfaceNormal(cpv.Parameters);
        double[,] Psurf = SurfaceProjection(Nsurf);

        for (int d = 0; d < cpv.D; d++)
            acc += - m_sigma * Psurf[m_comp, d] * GradV[d];

        return -acc;
    }

    /// InnerEdgeForm on $ \partial (K_j \cap \mathfrak{I})$
    public double InnerEdgeForm(ref CommonParams inp, double[] _uA, double[] _uB, double[,] _Grad_uA, double[,] _Grad_uB, double _vA, double _vB, double[] _Grad_vA, double[] _Grad_vB) {

        double[] EdgeNormal = inp.Normal;
        double[] SurfaceNormal_IN = SurfaceNormal(inp.Parameters_IN);
        double[] SurfaceNormal_OUT = SurfaceNormal(inp.Parameters_OUT);

        double[] Tangente_IN = Tangent(SurfaceNormal_IN, EdgeNormal);
        double[] Tangente_OUT = Tangent(SurfaceNormal_OUT, EdgeNormal);

        double acc = m_sigma *  0.5 * (Tangente_IN[m_comp] + Tangente_OUT[m_comp]) * (_vA - _vB);

        return -acc;
    }

    /// Since we are only considering closed interfaces, we don't need to implement a BoundaryEdgeForm
    public double BoundaryEdgeForm(ref CommonParamsBnd inp, double[] _uA, double[,] _Grad_uA, double _vA, double[] _Grad_vA) {
        throw new NotImplementedException("interface should always be closed");
    }


    /// surface normal pointing fomr A to B
    protected static double[] SurfaceNormal(double[] param) {

        double[] N = new double[param.Length];

        for (int d = 0; d < param.Length; d++) {
            N[d] = param[d];
        }

        return N.Normalize();
    }

    /// P_\mathfrak{I} = \mathbf{I} - \vec{n}_\mathfrak{I} \otimes \vec{n}_\mathfrak{I}
    protected static double[,] SurfaceProjection(double[] Nsurf) {

        int D = Nsurf.Length;
        double[,] P = new double[D, D];

        for (int d = 0; d < D; d++) {
            for (int dd = 0; dd < D; dd++) {
                if (dd == d)
                    P[d, dd] = (1 - Nsurf[d] * Nsurf[dd]);
                else
                    P[d, dd] = (0 - Nsurf[d] * Nsurf[dd]);
            }
        }

        return P;
    }

    /// compute the tangent at the cell boundary 
    protected static double[] Tangent(double[] Nsurf, double[] Nedge) {

        int D = Nsurf.Length;

        double[] tau = new double[D];
        for (int d1 = 0; d1 < D; d1++) {
            for (int d2 = 0; d2 < D; d2++) {
                double nn = Nsurf[d1] * Nsurf[d2];
                if (d1 == d2) {
                    tau[d1] += (1 - nn) * Nedge[d2];
                } else {
                    tau[d1] += -nn * Nedge[d2];
                }
            }
        }

        return tau.Normalize();
    }


    public virtual IList<string> ParameterOrdering {
        get {
            return VariableNames.NormalVector(m_D);
        }
    }

    public IList<string> ArgumentOrdering {
        get {
            return VariableNames.VelocityVector(m_D);
        }
    }

    public TermActivationFlags VolTerms {
        get {
            return TermActivationFlags.GradV;
        }
    }

    public TermActivationFlags InnerEdgeTerms {
        get {
            return TermActivationFlags.V;
        }
    }

    public TermActivationFlags BoundaryEdgeTerms {
        get {
            return TermActivationFlags.V | TermActivationFlags.UxV; 
        }
    }


}

For the definition of the `SpatialOperator`, we need to define and add some parameter fields. The following code is just an helper class for the following construction. 

In [ ]:
public class ParameterList {

    public void VerifyList(IEnumerable<string> parameterNames) {
        foreach(var p in this.parameters) {
            foreach(string OtherPname in p.ParameterNames) {
                if(!parameterNames.Contains(OtherPname)) {
                    //throw new ArgumentException($"Smells like configuration error: an updater for parameter {OtherPname} was added, but none of the equations seem to need this parameter.");
                    //Console.WriteLine($"Warning: smells like configuration error: an updater for parameter {OtherPname} was added, but none of the equations seem to need this parameter.");
                }
            }
        }

        foreach(var OtherPname in parameterNames) {
            bool bfound = false;
            foreach(var p in this.parameters) {
                if(p.ParameterNames.Contains(OtherPname))  {
                    bfound = true;
                    break;
                }
            }
            if(!bfound)
                //Console.Error.WriteLine($"Smells like configuration error: an parameter {OtherPname} is specified by the operator, but none of the updaters seems to care about this parameter.");
                throw new ArgumentException($"Smells like configuration error: an parameter {OtherPname} is specified by the operator, but none of the updaters seems to care about this parameter.");
        }

    }

    List<ParameterS> parameters;

    public ParameterList(int capacity = 10) {
        parameters = new List<ParameterS>(capacity);
    }

    public void AddParameter(ParameterS parameter) {
        parameters.Add(parameter);
    }

    public ICollection<DelParameterFactory> Factories(IList<string> names) {
        LinkedList<string> nameList = new LinkedList<string>(names);
        LinkedList<DelParameterFactory> parameterFactories = new LinkedList<DelParameterFactory>();
        //Find parameters and remove all found parameters from list;

        while(nameList.Count > 0) {
            string name = nameList.First.Value;
            nameList.RemoveFirst();
            //Find currentName
            for(int i = 0; i < parameters.Count; ++i) {
                ParameterS parameter = parameters[i];
                if(parameter.ParameterNames.Contains(name)) {
                    if(parameter.Factory != null) {
                        parameterFactories.AddLast(parameter.Factory);
                    }
                    foreach(string otherParamName in parameter.ParameterNames) {
                        nameList.Remove(otherParamName);
                    }
                    break;
                }
            }
        }
        return parameterFactories;
    }

    public ICollection<DelPartialParameterUpdate> ParameterUpdates(IList<string> names) {
        LinkedList<string> nameList = new LinkedList<string>(names);
        LinkedList<DelPartialParameterUpdate> parameterUpdates = new LinkedList<DelPartialParameterUpdate>();

        //Find parameters and remove all found parameters from list;
        while(nameList.Count > 0) {
            string name = nameList.First.Value;
            nameList.RemoveFirst();
            //Find currentName
            for(int i = 0; i < parameters.Count; ++i) {
                ParameterS parameter = parameters[i];
                if(parameter.ParameterNames.Contains(name)) {
                    if(parameter.Update != null) {
                        parameterUpdates.AddLast(parameter.Update);
                    }
                    foreach(string otherParamName in parameter.ParameterNames) {
                        nameList.Remove(otherParamName);
                    }
                    break;
                }
            }
        }
        return parameterUpdates;
    }
}

Now, we define our own solver class as a derivative from `XNSE`.

In [ ]:
public class NavierStokesXDGSolver : XNSE {

    static void Main(string[] args) {
        _Main(args, false, delegate () {
            var p = new NavierStokesXDGSolver();
            return p;
        });
    }

    // override in order to remove old .plt-files.
    public override void Init(AppControl control) {

        // remove old plot files
        var dir = new DirectoryInfo(Directory.GetCurrentDirectory() + "/plots");
        Console.Write("rm");
        foreach (var pltFile in dir.GetFiles("NavierStokesXDGSolver-*")) {
            Console.Write(" " + pltFile.Name);
            pltFile.Delete();
        }
        Console.WriteLine(";");

        base.Init(control);
    }

    // override in order to save the .plt-files in a subfolder. Furthermore, one may add additional fields to be plotted. 
    protected override void PlotCurrentState(double physTime, TimestepNumber timestepNo, int superSampling = 0) {
        BoSSS.Solution.Tecplot.Tecplot.PlotFields(this.m_RegisteredFields, "plots/NavierStokesXDGSolver-" + timestepNo, physTime, superSampling);
    }

    // override Operator/equation assembly
    protected override XSpatialOperatorMk2 GetOperatorInstance(int D, LevelSetUpdater levelSetUpdater) {
   
        // define some paramaters for the instantiation of the operator
        // ============================================================
        string[] gravitySpeciesVector = new string[] {"GravityX#A", "GravityX#B", "GravityY#A", "GravityY#B"};
        string[] parameterVars = (this.Control.PhysicalParameters.IncludeConvection) ? 
        BoSSS.Solution.NSECommon.VariableNames.Velocity0Vector(D).Cat(BoSSS.Solution.NSECommon.VariableNames.Velocity0MeanVector(D)).Cat(gravitySpeciesVector).Cat(BoSSS.Solution.NSECommon.VariableNames.NormalVector(D)) 
        : gravitySpeciesVector.Cat(BoSSS.Solution.NSECommon.VariableNames.NormalVector(D));

        string[] coDomainVars = (new[] { "ResidualMomentumX", "ResidualMomentumY" }).Cat("ResidualConti");

        int quadOrder = QuadOrder();
        int QuadOrderFunc(int[] DomvarDegs, int[] ParamDegs, int[] CodvarDegs) {
            return quadOrder;
        };

        string[] species = new string[] {"A", "B"};

        // instantiate operator
        // ====================
        XSpatialOperatorMk2 XOP = new XSpatialOperatorMk2(
            __DomainVar: VariableNames.VelocityVector(D).Cat(VariableNames.Pressure),
            __ParameterVar: parameterVars,
            __CoDomainVar: coDomainVars,
            QuadOrderFunc,
            __Species: species);


        
        GetBcMap();     // instantiate boundary condition mapping

        ParameterList Parameters = new ParameterList();

        var physParams = this.Control.PhysicalParameters;


        // =======================================
        // add equation components to the operator
        // =======================================


        // bulk momentum equations
        // =======================
        for (int d = 0; d < D; ++d) { 
            foreach (string spc in species) {

            // set species arguments
            double rhoSpc, LFFSpc, muSpc;
            switch (spc) {
                case "A": { rhoSpc = physParams.rho_A; LFFSpc = this.Control.AdvancedDiscretizationOptions.LFFA; muSpc = physParams.mu_A; break; }
                case "B": { rhoSpc = physParams.rho_B; LFFSpc = this.Control.AdvancedDiscretizationOptions.LFFB; muSpc = physParams.mu_B; break; }
                default: throw new ArgumentException("Unknown species.");
            }

            // convective operator
            // ===================
            if (this.Control.PhysicalParameters.IncludeConvection) {
                var conv = new BoSSS.Solution.XNSECommon.Operator.Convection.ConvectionInSpeciesBulk_LLF(D, boundaryMap, spc, d, rhoSpc, LFFSpc);
                XOP.EquationComponents[coDomainVars[d]].Add(conv);
            }


            // pressure gradient
            // =================
            var pres = new BoSSS.Solution.XNSECommon.Operator.Pressure.PressureInSpeciesBulk(d, boundaryMap, spc);
            XOP.EquationComponents[coDomainVars[d]].Add(pres);
            


            // viscous operator
            // ================
            var Visc1 = new BoSSS.Solution.XNSECommon.Operator.Viscosity.ViscosityInSpeciesBulk_GradUTerm(
                this.Control.AdvancedDiscretizationOptions.PenaltySafety, 1.0,
                boundaryMap, spc, d, D, physParams.mu_A, physParams.mu_B);
            XOP.EquationComponents[coDomainVars[d]].Add(Visc1);

            var Visc2 = new BoSSS.Solution.XNSECommon.Operator.Viscosity.ViscosityInSpeciesBulk_GradUtranspTerm(
                this.Control.AdvancedDiscretizationOptions.PenaltySafety, 1.0, 
                boundaryMap, spc, d, D, physParams.mu_A, physParams.mu_B);
            XOP.EquationComponents[coDomainVars[d]].Add(Visc2);


            // gravity source term
            // ===================
            string gravity = BoSSS.Solution.NSECommon.VariableNames.GravityVector(D)[d];
            string gravityOfSpecies = gravity + "#" + spc;
            var gravityComponent = new BoSSS.Solution.XNSECommon.Operator.MultiPhaseSource(gravityOfSpecies, spc);
            XOP.EquationComponents[coDomainVars[d]].Add(gravityComponent);
            var GravParam = Gravity.CreateFrom(spc, d, D, Control, rhoSpc, Control.GetGravity(spc, d));
            Parameters.AddParameter(GravParam);

            }
        }

        // bulk continuity equation
        // ========================
        for (int d = 0; d < D; ++d) { 
            foreach (string spc in species) {

            // set species arguments
            double rhoSpc, LFFSpc, muSpc;
            switch (spc) {
                case "A": { rhoSpc = physParams.rho_A; LFFSpc = this.Control.AdvancedDiscretizationOptions.LFFA; muSpc = physParams.mu_A; break; }
                case "B": { rhoSpc = physParams.rho_B; LFFSpc = this.Control.AdvancedDiscretizationOptions.LFFB; muSpc = physParams.mu_B; break; }
                default: throw new ArgumentException("Unknown species.");
            }

            var src = new BoSSS.Solution.XNSECommon.Operator.Continuity.DivergenceInSpeciesBulk_Volume(d, D, spc, rhoSpc, -1, false);
            XOP.EquationComponents[coDomainVars[D]].Add(src);

            var flx = new BoSSS.Solution.XNSECommon.Operator.Continuity.DivergenceInSpeciesBulk_Edge(d, boundaryMap, spc, rhoSpc, -1, false);
            XOP.EquationComponents[coDomainVars[D]].Add(flx);

            }
        }


        // interface momentum equations 
        // ============================
        for (int d = 0; d < D; ++d) { 

            // convective term
            // ===============
            if (this.Control.PhysicalParameters.IncludeConvection) {
                var convLS = new BoSSS.Solution.XNSECommon.Operator.Convection.ConvectionAtLevelSet_LLF(d, D, physParams.rho_A, physParams.rho_B, 
                    this.Control.AdvancedDiscretizationOptions.LFFA, this.Control.AdvancedDiscretizationOptions.LFFB, _MaterialInterface: true, boundaryMap, _movingmesh: false);
                XOP.EquationComponents[coDomainVars[d]].Add(convLS);
            }

            // pressure term
            // =============
            var presLs = new BoSSS.Solution.XNSECommon.Operator.Pressure.PressureFormAtLevelSet(d, D);
            XOP.EquationComponents[coDomainVars[d]].Add(presLs);

            // viscosity term
            // ==============
            var viscLS = new ViscosityAtLevelSet(D, physParams.mu_A, physParams.mu_B, this.Control.AdvancedDiscretizationOptions.PenaltySafety, d);
            // var viscLS = new BoSSS.Solution.XNSECommon.Operator.Viscosity.ViscosityAtLevelSet_FullySymmetric(D, physParams.mu_A, physParams.mu_B, 
            //     this.Control.AdvancedDiscretizationOptions.PenaltySafety, d, false);
            //var viscLS = new BoSSS.Solution.XNSECommon.Operator.Viscosity.ViscosityAtLevelSet_Standard(muA, muB, penalty * 1.0, D, d, false);
            XOP.EquationComponents[coDomainVars[d]].Add(viscLS);

            // surface tension force
            // =====================
            var isoSurfT = new SurfaceTensionForce(d, D, physParams.Sigma * 0.5);
            //var isoSurfT = new BoSSS.Solution.XNSECommon.Operator.SurfaceTension.IsotropicSurfaceTension_LaplaceBeltrami(d, D, physParams.Sigma * 0.5, boundaryMap.EdgeTag2Type, boundaryMap, physParams.theta_e, physParams.betaL);
            XOP.SurfaceElementOperator_Ls0.EquationComponents[coDomainVars[d]].Add(isoSurfT);

        }

        // interface continuity equation
        // =============================
        var divPen = new BoSSS.Solution.XNSECommon.Operator.Continuity.DivergenceAtLevelSet(D, physParams.rho_A, physParams.rho_B, _MaterialInterface: true, -1, false);
        XOP.EquationComponents[coDomainVars[D]].Add(divPen);

        
        // temporal term
        // =============
        (string, double[])[] diagonal = new (string, double[])[species.Length];
        diagonal[0] = ("A", new double[] {1, 1, 0});
        diagonal[1] = ("B", new double[] {1, 1, 0});
        XOP.TemporalOperator = new ConstantXTemporalOperator(XOP, diagonal);


        // additional parameters
        // =====================
        Velocity0Mean v0Mean = new Velocity0Mean(D, LsTrk, quadOrder);
        if (this.Control.PhysicalParameters.IncludeConvection) {
            Parameters.AddParameter(new Velocity0(D));
            Parameters.AddParameter(v0Mean);
        }
        Normals normalsParameter = new Normals(D, ((LevelSet)levelSetUpdater.Tracker.LevelSets[0]).Basis.Degree);
        Parameters.AddParameter(normalsParameter);

        ICollection<DelParameterFactory> factories = Parameters.Factories(XOP.ParameterVar);
        foreach (DelParameterFactory factory in factories) {
            XOP.ParameterFactories.Add(factory);
        }
        ICollection<DelPartialParameterUpdate> updates = Parameters.ParameterUpdates(XOP.ParameterVar);
        foreach (DelPartialParameterUpdate update in updates) {
            XOP.ParameterUpdates.Add(update);
        }

        // level set related parameters 
        // ============================
        levelSetUpdater.AddLevelSetParameter(VariableNames.LevelSetCG, v0Mean);
        levelSetUpdater.AddLevelSetParameter(VariableNames.LevelSetCG, normalsParameter);
        levelSetUpdater.AddLevelSetParameter(VariableNames.LevelSetCG, FromControl.BeltramiGradient(Control, "Phi", D));


        // coefficients
        // ============
        XOP.OperatorCoefficientsProvider = delegate (LevelSetTracker lstrk, SpeciesId spc, int quadOrder, int TrackerHistoryIdx, double time) {
            CoefficientSet CoeffSet = new CoefficientSet() {
                GrdDat = lstrk.GridDat
            };
            CoeffSet.CellLengthScales = ((BoSSS.Foundation.Grid.Classic.GridData)CoeffSet.GrdDat).Cells.CellLengthScale;
            CoeffSet.EdgeLengthScales = ((BoSSS.Foundation.Grid.Classic.GridData)CoeffSet.GrdDat).Edges.h_min_Edge;   
            CoeffSet.SpeciesSubGrdMask = lstrk.Regions.GetSpeciesSubGrid(spc).VolumeMask.GetBitMaskWithExternal();

            return CoeffSet;
        };


        // final settings
        // ==============

        // if there is no Dirichlet boundary condition,
        // the mean value of the pressure is free:
        XOP.FreeMeanValue[VariableNames.Pressure] = !GetBcMap().DirichletPressureBoundary;

        XOP.IsLinear = !this.Control.PhysicalParameters.IncludeConvection;
        XOP.LinearizationHint = LinearizationHint.AdHoc;
        XOP.AgglomerationThreshold = this.Control.AgglomerationThreshold;


        XOP.Commit();

        return XOP;
    }

}

Define a corresponding control class

In [ ]:
[Serializable]
[DataContract]
public class NavierStokesdXDGSolverControl : XNSE_Control {
   
    /// <summary>
    /// required for BoSSSpad workflow management
    /// </summary>
    public override Type GetSolverType() {
        return typeof(NavierStokesXDGSolver);
    }
}

## Setup test cases

### Test case 1 - static droplet 

In [ ]:
NavierStokesdXDGSolverControl C1 = new NavierStokesdXDGSolverControl();

// database and saving options
C1.DbPath = null;
C1.savetodb = false;
C1.ProjectName = "StatciDroplet";
C1.ImmediatePlotPeriod = 1;
C1.SuperSampling = 3;

// Physical parameters
C1.PhysicalParameters.rho_A = 1.0;
C1.PhysicalParameters.rho_B = 1.0;
C1.PhysicalParameters.mu_A = 1.0;
C1.PhysicalParameters.mu_B = 1.0;
C1.PhysicalParameters.Sigma = 1.0;

C1.PhysicalParameters.IncludeConvection = false;
C1.PhysicalParameters.Material = true;

// DG degree
C1.SetDGdegree(3);

// computational grid
C1.GridFunc = delegate {

    double size = 1.0;
    int gridResolution = 8;
    double[] _Nodes = GenericBlas.Linspace(-size, size, gridResolution * 2 + 1);
    var grd = Grid2D.Cartesian2DGrid(_Nodes, _Nodes, BoSSS.Foundation.Grid.RefElements.CellType.Square_Linear);

    grd.DefineEdgeTags(delegate (double[] _X) {
        double x = _X[0];
        double y = _X[1];

    if(Math.Abs(y - (-size)) < 1.0e-6       // bottom
        || Math.Abs(y - (+size)) < 1.0e-6   // top
        || Math.Abs(x - (-size)) < 1.0e-6   // left
        || Math.Abs(x - (size)) < 1.0e-6)   // right
        return IncompressibleBcType.Wall.ToString();
    else
        throw new ArgumentOutOfRangeException();
    });

    return grd;
};

// Set initial level-set, resp. interface position
C1.AddInitialValue("Phi", new Formula("X => ((X[0]).Pow2() + X[1].Pow2()).Sqrt() - 0.5", TimeDep: false));  // signed-distance
//C1.AddInitialValue("Phi", new Formula("X => ((X[0]).Pow2() + X[1].Pow2()) - (0.5).Pow(2)", TimeDep: false));  // quadratic


// Solver Options
C1.TimesteppingMode = AppControl._TimesteppingMode.Steady;
C1.TimeSteppingScheme = BoSSS.Solution.XdgTimestepping.TimeSteppingScheme.ImplicitEuler;
C1.Timestepper_LevelSetHandling = BoSSS.Solution.XdgTimestepping.LevelSetHandling.None;
C1.Option_LevelSetEvolution = BoSSS.Solution.LevelSetTools.LevelSetEvolution.None;



In [ ]:
AppControlExtensions.Run(C1);

### Test case 2 - droplet in channel flow

In [ ]:
NavierStokesdXDGSolverControl C2 = new NavierStokesdXDGSolverControl();

// database and saving options
C2.DbPath = null;
C2.savetodb = false;
C2.ProjectName = "ChannelFlow";
C2.ImmediatePlotPeriod = 1;
C2.SuperSampling = 2;

// Physical parameters
C2.PhysicalParameters.rho_A = 1.0;
C2.PhysicalParameters.rho_B = 1.0;
C2.PhysicalParameters.mu_A = 1.0;
C2.PhysicalParameters.mu_B = 0.01;
C2.PhysicalParameters.Sigma = 0.1;

C2.PhysicalParameters.IncludeConvection = false;
C2.PhysicalParameters.Material = true;

// DG degree
C2.SetDGdegree(3);

// computational grid
C2.GridFunc = delegate {
    int gridResolution = 4;

    double Length = 5.0;
    double Height = 2.0;

    double[] _xNodes = GenericBlas.Linspace(0, Length, gridResolution * 5 + 1);
    double[] _yNodes = GenericBlas.Linspace(-Height/2.0, Height/2.0, gridResolution * 2 + 1);

    var grd = Grid2D.Cartesian2DGrid(_xNodes, _yNodes, BoSSS.Foundation.Grid.RefElements.CellType.Square_Linear);

    grd.DefineEdgeTags(delegate (double[] _X) {
        double x = _X[0];
        double y = _X[1];

    if(Math.Abs(y - (-Height/2.0)) < 1.0e-6)
        // bottom
        return IncompressibleBcType.Wall.ToString() + "_bottom";

    if(Math.Abs(y - (+Height/2.0)) < 1.0e-6)
        // top
        return IncompressibleBcType.Wall.ToString()+ "_top";


    if(Math.Abs(x - (0.0)) < 1.0e-6)
        // left
        return IncompressibleBcType.Velocity_Inlet.ToString();

    if(Math.Abs(x - (Length)) < 1.0e-6)
        // right
        return IncompressibleBcType.Pressure_Outlet.ToString();

        throw new ArgumentOutOfRangeException();
    });

    return grd;
};

// set boundary conditions
C2.AddBoundaryValue(IncompressibleBcType.Velocity_Inlet.ToString(), "VelocityX", new Formula("X => 1 - X[1] * X[1]", TimeDep: false));

// Set Initial Conditions
C2.AddInitialValue("VelocityX", new Formula("X => 0.0", TimeDep: false));

// Set initial level-set, resp. interface position
C2.AddInitialValue("Phi", new Formula("X => ((X[0] - 1.0).Pow2() + X[1].Pow2()).Sqrt() - 0.5", TimeDep: false));  // signed-distance

// Solver Options
C2.TimesteppingMode = AppControl._TimesteppingMode.Transient;
C2.TimeSteppingScheme = BoSSS.Solution.XdgTimestepping.TimeSteppingScheme.ImplicitEuler;
C2.Timestepper_LevelSetHandling = BoSSS.Solution.XdgTimestepping.LevelSetHandling.LieSplitting;
C2.Option_LevelSetEvolution = BoSSS.Solution.LevelSetTools.LevelSetEvolution.FastMarching;

C2.NoOfTimesteps = 20;
C2.dtFixed = 0.025;



In [ ]:
AppControlExtensions.Run(C2);